##Benyamin Askari
##Student ID: 00790065

# Task 1: Analysis of Clinical Trial Data Using Spark SQL


#  **Table of Contents**

### **1. Load Data**
- 1.1 Header Inspection  
- 1.2 Load the Dataset Properly with Schema

### **2. Exploratory Data Analysis (EDA)**
- 2.1 Visual Inspection of DataFrame  
- 2.2 Schema Inspection  
- 2.3 Summary Statistics  
- 2.4 Dataset Dimensions  
- 2.5 Missing Values Count per Column  
- 2.6 Value Counts of Key Categorical Fields  
- 2.7 Date Normalization and Overwriting Original Columns  
- 2.8 Identifying Rows Missing All Required Fields  

### **3. Registering Cleaned Data for SQL Queries**

---

### **4. SQL-Based Analytical Tasks**

####  **Question 1 — Most Frequent Study Types**
- Step 1: Create a Filtered View of Study Types with Frequency ≥ 8  
- Step 2: Total Count of Included Records
- Step 3: Display the Frequency Table 

####  **Question 2 — Top 10 Most Frequent Medical Conditions**
- Step 1: Split and Normalize Conditions  
- Step 2: Count Total Number of Contributing Condition Records 
- Step 3: Rank Top 10 Most Common Conditions 

####  **Question 3 — Mean Duration of Clinical Trials**
- Step 1: Filter for Valid Dates and Calculate Duration  
- Step 2: Count Contributing Records  
- Step 3: Compute Mean Duration in Months  

####  **Question 4 — Completed Diabetes-Related Trials by Year**
- Step 1: Filter and Extract Completion Year 
- Step 2: Count Contributing Records  
- Step 3: Count Per Year to Analyze Trend

##1. Load Data
In this section, we first inspect the raw CSV file as text to understand its structure before proceeding with schema inference and formal loading.

###1.1. Header Inspection
Before formally reading the CSV as a structured DataFrame, it is important to validate:

Whether the file contains a header row.

The delimiter used in the file (e.g., comma, pipe, etc.).

General data cleanliness.

**To do this, the raw CSV was read as plain text**

In [0]:
# Loading the raw CSV content as plain text to inspect its structure
raw_df = spark.read.text("/FileStore/tables/Clinicaltrial.csv")

In [0]:
# raw CSV content
raw_df.take(2)

Out[2]: [Row(value='NCT Number,Study Title,Acronym,Study Status,Conditions,Interventions,Sponsor,Collaborators,Enrollment,Funder Type,Study Type,Study Design,Start Date,Completion Date'),
 Row(value='NCT05013879,Kinesiotape for Edema After Bilateral Total Knee Arthroplasty,,COMPLETED,"Arthroplasty Complications|Arthroplasty, Replacement, Knee",DEVICE: Kinesio(R)Tape for edema control,Montefiore Medical Center,Burke Rehabilitation Hospital,65,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED|Intervention Model: SINGLE_GROUP|Masking: NONE|Primary Purpose: TREATMENT,2021-10-18,2023-11-24')]

The first row contains the column names (NCT Number, Study Title, Acronym, etc.).

The second row is actual data from a clinical trial.

Comma (,) is the separator.

**This confirms that the file does include a header, and is comma-separated as expected.**

###1.2. Load the Dataset Properly with Schema
After inspection, we now load the dataset properly:

Specify that the file has a header.

Ask Spark to infer the schema automatically based on data types.

In [0]:
# loading the datset
trials_df = spark.read\
    .option("header", True)\
    .option("inferSchema", True)\
    .csv("/FileStore/tables/Clinicaltrial.csv")


**This creates a structured DataFrame with:**

Properly labeled columns.

Automatically detected types like string, date, or integer based on the column contents.

##2. Exploratory Data Analysis (EDA)
This section aims to develop an initial understanding of the structure, completeness, and characteristics of the clinical trial dataset before moving forward to answering the core questions.

###2.1. Visual Inspection of DataFrame
To preview the dataset visually within the Databricks notebook, the following command was used.

This renders a scrollable, tabular interface showing column names and their sample values.

In [0]:
# data summary
display(trials_df)

NCT Number,Study Title,Acronym,Study Status,Conditions,Interventions,Sponsor,Collaborators,Enrollment,Funder Type,Study Type,Study Design,Start Date,Completion Date
NCT05013879,Kinesiotape for Edema After Bilateral Total Knee Arthroplasty,null,COMPLETED,"Arthroplasty Complications|Arthroplasty, Replacement, Knee",DEVICE: Kinesio(R)Tape for edema control,Montefiore Medical Center,Burke Rehabilitation Hospital,65,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED|Intervention Model: SINGLE_GROUP|Masking: NONE|Primary Purpose: TREATMENT,2021-10-18,2023-11-24
NCT00517179,Effect of Vardenafil on Blood Pressure in Patients With Erectile Dysfunction Who Received Concomitant Doxazosin GITS,null,COMPLETED,Prostatic Hyperplasia|Impotence,DRUG: Vardenafil 10mg,"Hospital Authority, Hong Kong",null,40,OTHER_GOV,INTERVENTIONAL,Allocation: RANDOMIZED|Intervention Model: CROSSOVER|Masking: DOUBLE|Primary Purpose: TREATMENT,2006-04,2007-05
NCT06714279,Laparoscopic-Assisted Transversus Abdominus Plane Block Versus Intraperitoneal Irrigation of Local Anesthetic for Patients Undergoing Laparoscopic Cholecystectomy,null,NOT_YET_RECRUITING,Laparoscopic Cholecystectomy|TAP Block|Local Anesthetic,DRUG: Tap Block - Bupivacaine|DRUG: Intraperitoneal infiltration to liver,"Royal College of Surgeons, Ireland",null,144,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED|Intervention Model: PARALLEL|Masking: NONE|Primary Purpose: TREATMENT,2025-01,2025-01
NCT05600179,OCTA in Epivascular Glia After Dex Implant,null,COMPLETED,Diabetic Retinopathy,DRUG: Dexamethasone intravitreal implant,Federico II University,null,38,OTHER,OBSERVATIONAL,Observational Model: |Time Perspective: p,2021-01-01,2022-09-30
NCT01511679,Brain-imaging and Adolescent Neuroscience Consortium,BANC,WITHDRAWN,Alcohol Abuse,null,Boston Children's Hospital,Massachusetts General Hospital|Mclean Hospital|Massachusetts Institute of Technology,0,OTHER,OBSERVATIONAL,Observational Model: |Time Perspective: p,2012-09,2017-09
NCT05602779,Leverage Noninvasive Transcutaneous Vagus Nerve Stimulation to Reduce Suicidal Behaviors in Vulnerable Adolescents,null,RECRUITING,Self Harm|Suicidal Ideation,DEVICE: tVns Program|OTHER: Phone App Program|COMBINATION_PRODUCT: tVNS and Phone App Program|OTHER: Enhanced Treatment as Usual,University of Notre Dame,University of Rochester,212,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED|Intervention Model: PARALLEL|Masking: SINGLE (PARTICIPANT)|Primary Purpose: PREVENTION,2023-10-08,2027-09-30
NCT04175379,The Effect of Permissive Hypercapnia on Oxygenation and Post-operative Pulmonary Complication During One-lung Ventilation,null,UNKNOWN,Thoracic Surgery,OTHER: group 40|OTHER: group 50|OTHER: group 60,Yonsei University,null,279,OTHER,INTERVENTIONAL,"Allocation: RANDOMIZED|Intervention Model: PARALLEL|Masking: TRIPLE (PARTICIPANT, CARE_PROVIDER, OUTCOMES_ASSESSOR)|Primary Purpose: TREATMENT",2019-11-25,2021-10
NCT01126879,Genistein in Treating Patients With Prostate Cancer,null,TERMINATED,Adenocarcinoma of the Prostate|Recurrent Prostate Cancer|Stage I Prostate Cancer|Stage II Prostate Cancer|Stage III Prostate Cancer,DIETARY_SUPPLEMENT: genistein|OTHER: placebo|PROCEDURE: therapeutic conventional surgery,Northwestern University,National Cancer Institute (NCI),12,OTHER,INTERVENTIONAL,"Allocation: RANDOMIZED|Intervention Model: PARALLEL|Masking: DOUBLE (PARTICIPANT, INVESTIGATOR)|Primary Purpose: TREATMENT",2011-02-03,2013-12-28
NCT03058679,Trial of Specific Carbohydrate and Mediterranean Diets to Induce Remission of Crohn's Disease,DINE-CD,COMPLETED,Crohn Disease,OTHER: Diet,University of Pennsylvania,"Patient-Centered Outcomes Research Institute|Crohn's and Colitis Foundation|University of North Carolina, Chapel Hill",197,OTHER,INTERVENTIONAL,Allocation: RANDOMIZED|Intervention Model: PARALLEL|Masking: NONE|Primary Purpose: TREATMENT,2017-09-29,2020-03-01
NCT05531279,A Study of PEG-rhG-CSF and rhG-CSF Used for Aplastic Anemia Granulocyte Deficiency,null,RECRUITING,Severe Aplastic Anemi

**So far, the datset apears to have a lot of missing and invalid values. it is up to the following analysis to decide how to deal with them**

###2.2. Schema Inspection

In [0]:
# Importing required libraries for the data exploration
from pyspark.sql.functions import (
    col, count, when, split, explode, months_between,
    to_date, year, trim
)

Before performing computations, it's essential to verify the structure of the DataFrame.

**This includes checking:**

Number of columns

Column names

Data types

Nullability of fields

To do this, we call .printSchema() on the DataFrame.

In [0]:
# Schema and basic statistics
trials_df.printSchema()

root
 |-- NCT Number: string (nullable = true)
 |-- Study Title: string (nullable = true)
 |-- Acronym: string (nullable = true)
 |-- Study Status: string (nullable = true)
 |-- Conditions: string (nullable = true)
 |-- Interventions: string (nullable = true)
 |-- Sponsor: string (nullable = true)
 |-- Collaborators: string (nullable = true)
 |-- Enrollment: string (nullable = true)
 |-- Funder Type: string (nullable = true)
 |-- Study Type: string (nullable = true)
 |-- Study Design: string (nullable = true)
 |-- Start Date: string (nullable = true)
 |-- Completion Date: string (nullable = true)



All columns are of string type including dates and numeric fields like Enrollment.

Spark inferred all fields as nullable, which is standard but reinforces the need to check for missing values in the next step.

Dates will need to be explicitly cast to date type for time-based analysis such as calculating duration (Q3).

###2.3. Summary Statistics
To gain a quick overview of the dataset’s numerical and string-based characteristics, we use the built-in describe()**** function. While primarily designed for numerical values, it still gives us thw count of non-null entries per column, min, max values (useful for string sorting or ID ranges), and mean, stddev (only when numeric).

In [0]:
# summary stats for numeric columns
display(trials_df.describe())

summary,NCT Number,Study Title,Acronym,Study Status,Conditions,Interventions,Sponsor,Collaborators,Enrollment,Funder Type,Study Type,Study Design,Start Date,Completion Date
count,522660,522660,145486,522611,521707,470997,522650,169184,515612,521715,521741,520791,517470,505988
mean,null,null,Infinity,null,null,137.0,null,null,5455.097777010285,1893.189349112426,877.9101123595506,8175.351351351352,97.16666666666667,383.1666666666667
stddev,null,null,NaN,null,null,null,null,null,478064.21568280784,11075.424761267846,2919.3776408697477,46886.72448940473,94.48897642935216,792.4540154902786
min,NCT00000102,""""""" Acute Brain Changes After Repetitive Headers in Soccer and the Effects of a Protective Device """"""","""""BR3006-2""""","& Breathe"""" Nursing Intervention For Patients With Esophageal Cancer""","Cluster-randomized Trial Within an Established Collaborative""",Inulin Capsule,"""""Celebrex®""""""","""""Epinephrine""""","""""Morphine"""" and """"Surgicel""""|DRUG: """"Normal Saline""""""","""""Dorifen®""""; """"Fentanyl""""","""""Talinat®""""","''Patient Controlled Analgesia Device''""","""""You are safe here""""",Austria|Spedali Civili
max,NCT06777485,⁸⁹Zr-Df-IAB22M2C PET/CT in Patients With Selected Solid Malignancies or Hodgkin's Lymphoma,单孔机器人-SPR,WITHHELD,•Non-alcoholic Steatohepatitis (NASH),WITHDRAWN,池畔,Şerife İrem DÖNER|Merve YAZAR|JULE ERİÇ HORASANLI,Uppsala University|Karolinska Institutet|The University of Western Australia,Zosano Pharma Corporation,University Ghent,The state institution N. N. Alexandrov National Cancer Centre of Belarus|Unitary Enterprise UNITEHPROM BSU,delta medical promotions ag,Observational Model: |Time Perspective: p


The column Acronym only has ~145k rows, suggesting high missingness, something to address in the next step.

Fields like Study Title and Study Status appear mostly populated and will be useful for analysis.

###2.4. Dataset Dimensions
Before performing any analysis, it is useful to understand the overall shape of the dataset like:

Total number of rows (records)

Total number of columns (fields per record)

In [0]:
# Total rows
trials_df.count()

Out[9]: 522660

In [0]:
# Total columns
len(trials_df.columns)

Out[8]: 14

###2.5. Missing Values Count per Column
To ensure data quality, it is essential to identify columns that have missing or empty ("") values. These nulls must be understood before running any transformations or SQL queries, especially when columns are central to answering project questions.

In [0]:
# Missing values count per column
missing_exprs = [
    count(when(col(c).isNull() | (col(c) == ""), c)).alias(c)
    for c in trials_df.columns
]
missing_df = trials_df.select(*missing_exprs)
display(missing_df)

NCT Number,Study Title,Acronym,Study Status,Conditions,Interventions,Sponsor,Collaborators,Enrollment,Funder Type,Study Type,Study Design,Start Date,Completion Date
0,0,377174,49,953,51663,10,353476,7048,945,919,1869,5190,16672


Fields like NCT Number, Study Title, and Study Status are almost completely filled — these will form reliable axes for our grouping and filtering later.

Some fields are only relevant to specific trials (e.g., Acronym, Collaborators) and thus nullable by design.

For core columns used in the four analysis questions (like Study Type, Conditions, Completion Date), missing data will need to be filtered or handled.

###2.6. Value Counts of Key Categorical Fields
To understand the most common values in key fields and guide filtering and grouping operations for SQL tasks later, we count frequency distributions for:

**Study Type**

**Study Status**

**Funder Type**

In [0]:
#  Value counts of key categorical fields
display(trials_df.groupBy("Study Type").count().orderBy(col("count").desc()))
display(trials_df.groupBy("Study Status").count().orderBy(col("count").desc()))
display(trials_df.groupBy("Funder Type").count().orderBy(col("count").desc()))

Study Type,count
INTERVENTIONAL,399654
OBSERVATIONAL,120816
EXPANDED_ACCESS,966
null,919
OTHER,141
INDUSTRY,21
OTHER_GOV,8
60,5
150,5
30,5


Study Status,count
COMPLETED,285114
UNKNOWN,74230
RECRUITING,66839
TERMINATED,30353
NOT_YET_RECRUITING,22857
ACTIVE_NOT_RECRUITING,20601
WITHDRAWN,14806
ENROLLING_BY_INVITATION,4229
SUSPENDED,1643
WITHHELD,900


Funder Type,count
OTHER,367120
INDUSTRY,119482
OTHER_GOV,13552
NIH,11347
FED,4676
NETWORK,4609
null,945
INDIV,570
UNKNOWN,77
100,13


**For all columns, there are several anomalous values are present and will need to be filtered out before aggregations**

###2.7 Date Normalization
Clinical trials in this dataset have Start Date and Completion Date recorded using inconsistent formats like **yyyy-MM**, **dd/MM/yyyy**, or **yyyy / MM / dd**. These must be normalized before calculating trial durations.

The code below uses PySpark abd Removes all delimiters, Splits parts and identifies Year, Month, Day. Also, falls back to the 1st day when it's missing.

Then extracts year, month, day no matter their order and constructs a clean DateType column using **make_date()**. This creates a proper date safely for Spark operations.

we also overwrite the original Start Date and Completion Date columns directly with fully parsed, standardized Spark DateType columns.

In [0]:
from pyspark.sql.functions import col, regexp_replace, trim, split, expr, lit, when
from pyspark.sql.types import IntegerType
import pyspark.sql.functions as F

# Overwrite original Start Date and Completion Date columns
def normalize_date_column(df, input_col):
    """
    Normalize and overwrite a mixed-format date column in place.
    Supports formats like:
    - yyyy-MM-dd, yyyy-MM, yyyy/MM, yyyy MM
    - dd/MM/yyyy, dd / MM / yyyy
    - Handles spaces, slashes, dashes
    """
    cleaned_col = "cleaned_" + input_col.replace(" ", "_")

    # Step 1: Remove all non-digits and standardize delimiters as space
    df = df.withColumn(cleaned_col, regexp_replace(trim(col(input_col)), "[^0-9]", " "))
    df = df.withColumn("parts", split(col(cleaned_col), " "))
    df = df.withColumn("parts", expr("filter(parts, x -> x != '')"))

    # Step 2: Extract parts and infer year-month-day positions
    df = df.withColumn("part1", col("parts").getItem(0).cast(IntegerType()))
    df = df.withColumn("part2", col("parts").getItem(1).cast(IntegerType()))
    df = df.withColumn("part3", col("parts").getItem(2).cast(IntegerType()))

    df = df.withColumn("year", when(col("part1") > 1900, col("part1"))
                                 .when(col("part3") > 1900, col("part3"))
                                 .otherwise(None))

    df = df.withColumn("month", when(col("part1") > 1900, col("part2"))
                                  .when(col("part3") > 1900, col("part2"))
                                  .otherwise(col("part1")))

    df = df.withColumn("day", when(col("part1") > 1900, col("part3"))
                                .when(col("part3") > 1900, col("part1"))
                                .otherwise(lit(1)))

    # Step 3: Overwrite the original column with the parsed DateType version
    df = df.withColumn(input_col, F.expr("make_date(year, month, day)"))

    # Step 4: Clean up helper columns
    return df.drop(cleaned_col, "parts", "part1", "part2", "part3", "year", "month", "day")


Next, we update both columns in Spark’s DateType to ensure downstream calculations (like durations) work reliably.

In [0]:
# Apply in-place normalization
trials_df = normalize_date_column(trials_df, "Start Date")
trials_df = normalize_date_column(trials_df, "Completion Date")


After this, it uses Spark’s **months_between()** function to compute how long each clinical trial lasted

Results stored in new column: Duration_Months

Summary statistics (count, mean, min, max, quartiles) are calculated using .summary()

In [0]:
from pyspark.sql.functions import months_between

# Calculate trial length in months
duration_df = trials_df.withColumn("Duration_Months", months_between(col("End"), col("Start")))
display(duration_df.select("Start", "End", "Duration_Months").summary())


summary,Duration_Months
count,263599
mean,32.12214484171986
stddev,36.39258320984422
min,0.0
25%,10.64516129
50%,22.83870968
75%,41.93548387
max,1266.96774194


###2.8 Identifying Rows Missing All Required Fields
To check if there are any rows in the dataset that are completely missing all essential information required to answer any of the core questions (Q1–Q4). This is a defensive step to:

Prevent data skew during filtering

Validate data integrity after transformations (especially post date normalization)

In [0]:
# Identify rows missing *all* required columns for any question
# Define all columns used across the 4 questions
required_cols = [
    "Study Type",      # Q1
    "Conditions",      # Q2 & Q4
    "Start Date",      # Q3
    "Completion Date", # Q3 & Q4
    "Study Status"     # Q4
]
# Build predicate: True if every required column is null or empty
from functools import reduce
import operator

all_blank = reduce(
    operator.and_,
    [(col(c).isNull() | (col(c) == "")) for c in required_cols]
)

# Filter to those fully blank rows
totally_blank = trials_df.filter(all_blank)
# Display them and their count
display(totally_blank)
print(f"Rows missing every required field: {totally_blank.count()}")

NCT Number,Study Title,Acronym,Study Status,Conditions,Interventions,Sponsor,Collaborators,Enrollment,Funder Type,Study Type,Study Design,Start Date,Completion Date,Start,End


Rows missing every required field: 0


**Now, based on the missing value results, we decide to keep the rows with missing values to prevent affecting the columns that are totaly useful for a particular question and filter out missing values based on quesions.**

##3. Registering the Cleaned DataFrame for SQL Queries
the next step is to prepare the dataset for SQL-based analysis. This involves registering the trials_df DataFrame as a temporary SQL view. This line makes the trials_df available inside Spark SQL using the alias "trials"

In [0]:
# Register the existing DataFrame as a temp view
trials_df.createOrReplaceTempView("trials")

##Question 1 - Most Frequent Study Types

### Step 1: Create a Filtered View of Study Types with Frequency ≥ 8
As we saw in the EDA steps, the values more than 8 characters were invalide; therefore the threshold of 8 is set to only use the valid study types.

The code groups all records by type of study, filters out very rare or erroneous categories, ensures missing data is not counted, saves the result as a temporary SQL view for re-use

In [0]:
%sql
-- Create temp view of study types with at least 8 entries

CREATE OR REPLACE TEMP VIEW q1_filtered AS
SELECT `Study Type`, COUNT(*) AS frequency
FROM trials
WHERE `Study Type` IS NOT NULL
GROUP BY `Study Type`
HAVING COUNT(*) >= 8;

###Step 2: Total Count of Included Records

In [0]:
%sql
-- Calculates the total number of rows from the original 'trials'
SELECT SUM(frequency) AS contributed_rows_q1 FROM q1_filtered;

contributed_rows_q1
521606


This confirms that majority of the dataset is valid and usable for this analysis.

###Step 3: Display the Frequency Table
the code below uses the **q1_filtered** to count frequency and order the study types

In [0]:
%sql
-- Show all study types from 'q1_filtered', sorted by highest frequency

SELECT * FROM q1_filtered
ORDER BY frequency DESC;

Study Type,frequency
INTERVENTIONAL,399654
OBSERVATIONAL,120816
EXPANDED_ACCESS,966
OTHER,141
INDUSTRY,21
OTHER_GOV,8


Databricks visualization. Run in Databricks to view.

Based on the results, the majority of studies are **INTERVENTIONAL** and the least is **OTHER_GOV**

##Question 2 - Top 10 Most Frequent Medical Conditions
The "Conditions" column in the clinical trials dataset contains one or more medical conditions per trial, separated by pipes (|).

**The code below will:**

Split multivalued entries

Normalize each individual condition

Identify the top 10 most common medical conditions studied across all trials

### Step 1: Normalize and Split Conditions Column
The code flattens the array so each condition gets its own row and cleans whitespace from condition names. Then ensures blank/missing entries are excluded

In [0]:
%sql
-- Create temp view of individual, non-empty conditions split from the 'Conditions' column

CREATE OR REPLACE TEMP VIEW q2_filtered AS
SELECT TRIM(c) AS Condition
FROM trials
LATERAL VIEW explode(split(Conditions, '\\|')) AS c
WHERE Conditions IS NOT NULL AND Conditions <> '';

###Step 2: Count Total Number of Contributing Condition Records

In [0]:
%sql
-- Count total number of extracted conditions in 'q2_filtered'

SELECT COUNT(*) AS contributed_rows_q2 FROM q2_filtered;

contributed_rows_q2
914185


This confirms that over 914,000 condition records exist, after exploding multi-condition trials.

###Step 3: Rank Top 10 Most Common Conditions

In [0]:
%sql
-- Get top 10 most frequent non-empty conditions from 'q2_filtered'

SELECT Condition, COUNT(*) AS frequency
FROM q2_filtered
WHERE Condition IS NOT NULL AND Condition <> ''
GROUP BY Condition
ORDER BY frequency DESC
LIMIT 10;


Condition,frequency
Healthy,10308
Breast Cancer,7941
Obesity,6949
Stroke,4481
Hypertension,4256
Depression,4195
Prostate Cancer,4071
Pain,4055
HIV Infections,3818
Cancer,3529


Databricks visualization. Run in Databricks to view.

**Based on the results:**

Cancer-related conditions (Breast, Prostate, general Cancer) are heavily represented.

Cardiometabolic disorders like Obesity and Hypertension are equally dominant.

“Healthy” individuals show up strongly; indicating frequent inclusion of control groups.

##Question 3 - Mean Duration of Clinical Trials (Months)
Since the date normalization step overwrote the raw columns and parsed them into Spark DateType, we now rely directly on the cleaned "Start Date" and "Completion Date" columns.

###Step 1: Filter for Valid Dates and Calculate Duration
The code below, calculates the time in months between start and end dates, ensures we exclude trials with missing dates so we don't calculate incorrect durations, and 	saves this clean subset for later analysis and counting.

In [0]:
%sql
-- Create temp view with trials that have start and completion dates, adding duration in months

CREATE OR REPLACE TEMP VIEW q3_filtered AS
SELECT *,
  months_between(`Completion Date`, `Start Date`) AS Duration_Months
FROM trials
WHERE `Start Date` IS NOT NULL AND `Completion Date` IS NOT NULL;


The q3_normalized view now contains just a single column: the year each completed study ended.

###Step 2: Count Contributing Records
This tells us how many rows had valid Start Date and Completion Date fields and thus contributed to the duration calculation.

In [0]:
%sql
-- Count rows with valid start and completion dates in 'q3_filtered'

SELECT COUNT(*) AS contributed_rows_q3 FROM q3_filtered;


contributed_rows_q3
263599


this shows there are a lot of missing values in the two date columns

###Step 3: Compute Mean Duration in Months
The code uses **AVG(...)** to compute the arithmetic mean across all valid durations. the results are rounded to 2 decimal places for clarity

In [0]:
%sql
-- Calculate average trial duration (in months), rounded to 2 decimals

SELECT ROUND(AVG(Duration_Months), 2) AS average_duration_months
FROM q3_filtered;


average_duration_months
32.12


Databricks visualization. Run in Databricks to view.

**The average clinical trial lasts 32.12 months, just over 2.5 years**

##Question 4 - Completed Diabetes-Related Trials by Year

###Step 1: Filter and Extract Completion Year
The code below, Extracts the year from the cleaned Completion Date, Ensures we only analyze finished trials, Finds all entries that mention diabetes (case-sensitive match), and makes a Temporary view for reuse in counting and plotting.

In [0]:
%sql
-- Create temp view of completion years for completed diabetes-related studies

CREATE OR REPLACE TEMP VIEW q4_filtered AS
SELECT
  year(`Completion Date`) AS Completion_Year
FROM trials
WHERE `Completion Date` IS NOT NULL
  AND `Study Status` = 'COMPLETED'
  AND (Conditions LIKE '%Diabetes%' OR Conditions LIKE '%diabetes%');


###Step 2: Count Contributing Records
This gives us the number of diabetes-related, completed trials with a known end date.

In [0]:
%sql
-- Count completed diabetes-related studies with known completion years

SELECT COUNT(*) AS contributed_rows_q4 FROM q4_filtered;

contributed_rows_q4
5053


only **5053** studies meet the criteria

###Step 3: Count Per Year to Analyze Trend
The code below extracts **Completion_Year** from the parsed Completion Date then counts the number of records (trials) completed in each year. Afterwards the trials are grouped by year so counts are calculated. In the end, it ensures the output is sorted chronologically for trend visualization.

In [0]:
%sql
-- Count diabetes-related completed trials per year

SELECT Completion_Year, COUNT(*) AS diabetes_trials
FROM q4_filtered
GROUP BY Completion_Year
ORDER BY Completion_Year;

Completion_Year,diabetes_trials
1997,1
1998,1
1999,1
2001,3
2002,8
2003,19
2004,18
2005,14
2006,12
2007,17


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

As evident, the number of studies had remained steady until **2010** when it started to increase gradually. Then from **2013**, the increase became exponentially sharper and peaked at **2019** followed by fluctuations until **now**.